In [1]:
import re
import json

In [2]:
#input_file = "C:/Users/diogo/Desktop/perkier tech/ARAG/ARAG/artifacts/CODIGO_CIVIL_smaller.txt"
input_file = "C:/Users/diogo/Desktop/perkier tech/ARAG/ARAG/artifacts/CODIGO_CIVIL_RAW.txt"

In [3]:
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

In [4]:
text = re.sub(r"CÓDIGO CIVIL.*?CÓDIGO CIVIL", "CÓDIGO CIVIL", text, flags=re.S)

In [5]:
# Ensure fully uppercase words have a newline before and after
text = re.sub(r"(\s*)([A-ZÀ-ÖØ-Ý]{2,})(\s*)", r"\n\2\n", text)

In [6]:
# Replace multiple spaces and newlines with a single space
text = re.sub(r"\s+", " ", text)

# Restore newlines for better formatting
text = text.replace(". ", ".\n")

# Standardize common punctuation
text = text.replace("“", '"').replace("”", '"')  # Convert curly quotes to straight
text = text.replace("‘", "'").replace("’", "'")  # Convert curly apostrophes to straight
text = text.replace("—", "-")  # Convert em-dashes to simple dashes

# Trim leading/trailing spaces
text = text.strip()

In [7]:
# Fix numbered sections (e.g., "2.\nSome text" → "2. Some text")
text = re.sub(r"(\d+)\.\s*\n\s*", r"\1. ", text)

In [8]:
# Fix numbers followed by a hyphen (e.g., "2 -" → "2.")
text = re.sub(r"(\d+)\s*-\s*", r"\1. ", text)

In [9]:
# Ensure "Artigo X.º (Title)" is on its own line
text = re.sub(r"(Artigo \d+[.ººª]* \([^)]+\))", r"\n\1\n", text)

In [10]:
# Remove accidental newlines inside "Artigo X.º (Title)", preserving dots in the title
text = re.sub(r"(Artigo \d+[.ºª]* \([^\n]+)\n([^\n]+\))", r"\1 \2", text)

In [11]:
# Trim leading/trailing spaces
text = '\n'.join([line.lstrip() for line in text.splitlines()])

# Remove all blank lines
text = re.sub(r"\n+", "\n", text)

In [12]:
text = re.sub(r"(\s*)CAPÍTULO", r"\nCAPÍTULO", text)

In [13]:
text = re.sub(r"(\s*)Contém as alterações introduzidas pelos seguintes diplomas", r"\nContém as alterações introduzidas pelos seguintes diplomas", text)

In [14]:
text = re.sub(r"(\s*)Versões anteriores deste artigo:", r"\nVersões anteriores deste artigo:", text)

In [15]:
text = re.sub(r" SECÇÃO", r"\nSECÇÃO", text)

In [16]:
text = re.sub(r" SUBSECÇÃO", r"\nSUBSECÇÃO", text)

In [17]:
text = re.sub(r"(\s*)Artigo", r"\nArtigo", text)

In [18]:
text = re.sub(r"(\s*)TÍTULO", r"\nTÍTULO", text)

In [19]:
text = re.sub(r"(\s*)LIVRO", r"\nLIVRO", text)

In [20]:
text = '\n'.join([line for line in text.splitlines() if "Contém as alterações introduzidas pelos seguintes diplomas:" not in line])

In [21]:
text = '\n'.join([line for line in text.splitlines() if "Versões anteriores deste artigo:" not in line])

In [22]:
text = '\n'.join([line for line in text.splitlines() if "Revogado pelo Decreto-Lei" not in line])

In [23]:
# This regex matches a number followed by a dot, followed by a space, and then an uppercase word
text = re.sub(r"(\d+\.\s*[A-Z]+)", r"\n\1", text)

In [24]:
# Remove all blank lines
text = re.sub(r"\n+", "\n", text)

In [25]:
# Function to convert Roman numeral to integer
def roman_to_int(roman):
    roman_numerals = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    prev_value = 0
    
    for letter in reversed(roman):
        value = roman_numerals[letter]
        if value < prev_value:
            total -= value
        else:
            total += value
        prev_value = value
        
    return total

# Regex pattern to find Roman numerals after SECÇÃO, CAPÍTULO, or SUBSECÇÃO (case-insensitive)
pattern = r'(LIVRO|TÍTULO|SUBTÍTULO|CAPÍTULO|SECÇÃO|SUBSECÇÃO|DIVISÃO)(\s+)([IVXLCDM]+)'

# Function to replace Roman numerals with integers
def replace_roman_numbers(text):
    def replacement(match):
        # Convert Roman numeral to integer
        word = match.group(1)
        roman = match.group(3)
        normal_number = roman_to_int(roman)
        return f"{word} {normal_number}"
    
    # Replace the Roman numerals using the regex
    return re.sub(pattern, replacement, text)

# Convert the Roman numerals in the text
text = replace_roman_numbers(text)

In [26]:
print(text)

::: DL n.º 47344/66, de 25 de Novembro DL n.º 47344/66, de 25 de Novembro (versão actualizada) CÓDIGO CIVIL
LIVRO 1PARTE GERAL
TÍTULO 499 as leis, sua interpretação e aplicação
CAPÍTULO 1F ontes do direito
Artigo 1.º (Fontes imediatas)
1. São fontes imediatas do direito as leis e as normas corporativas.
2. Consideram-se leis todas as disposições genéricas provindas dos órgãos estaduais competentes; são normas corporativas as regras ditadas pelos organismos representativos das diferentes categorias morais, culturais, económicas ou profissionais, no domínio das suas atribuições, bem como os respectivos estatutos e regulamentos internos.
3. As normas corporativas não podem contrariar as disposições legais de carácter imperativo.
Artigo 3.º (Valor jurídico dos usos)
1. Os usos que não forem contrários aos princípios da boa fé são juridicamente atendíveis quando a lei o determine.
2. As normas corporativas prevalecem sobre os usos.
Artigo 4.º (Valor da equidade)  Os tribunais só podem resol

In [27]:
# Split based on the chapter headings
matches = re.split(r"(LIVRO \d+)", text)[1:]  # Skip empty first element if any

# Structure the data
livro_data = [
    {"text": matches[i + 1].strip(), "metadata": {"Livro": matches[i]}}
    for i in range(0, len(matches), 2)
]

livro_data

[{'text': 'PARTE GERAL\nTÍTULO 499 as leis, sua interpretação e aplicação\nCAPÍTULO 1F ontes do direito\nArtigo 1.º (Fontes imediatas)\n1. São fontes imediatas do direito as leis e as normas corporativas.\n2. Consideram-se leis todas as disposições genéricas provindas dos órgãos estaduais competentes; são normas corporativas as regras ditadas pelos organismos representativos das diferentes categorias morais, culturais, económicas ou profissionais, no domínio das suas atribuições, bem como os respectivos estatutos e regulamentos internos.\n3. As normas corporativas não podem contrariar as disposições legais de carácter imperativo.\nArtigo 3.º (Valor jurídico dos usos)\n1. Os usos que não forem contrários aos princípios da boa fé são juridicamente atendíveis quando a lei o determine.\n2. As normas corporativas prevalecem sobre os usos.\nArtigo 4.º (Valor da equidade)  Os tribunais só podem resolver segundo a equidade: a) Quando haja disposição legal que o permita; b) Quando haja acordo d

In [28]:
json_livro = livro_data[1]
livro = json_livro["metadata"]['Livro']
livro_text = json_livro["text"]

# Regular expression to match "Artigo X.º (Title)"
matches = re.split(r"(TÍTULO \d+)", livro_text)[1:]  # Skip empty first element if any

# Structure the data
titulo_data = [
    {
        "text": matches[i + 1].strip(),
        "metadata": {
            "Livro": livro,
            "Titulo": matches[i],
        },
    }
    for i in range(0, len(matches), 2)
]

titulo_data

[{'text': 'Das obrigações em geral\nCAPÍTULO 1 Disposições gerais\nSECÇÃO 1 Conteúdo da obrigação\nArtigo 397.º (Noção)\nObrigação é o vínculo jurídico por virtude do qual uma pessoa fica adstrita para com outra à realização de uma prestação.\nArtigo 398.º (Conteúdo da prestação)\n1. As partes podem fixar livremente, dentro dos limites da lei, o conteúdo positivo ou negativo da prestação.\n2. A prestação não necessita de ter valor pecuniário; mas deve corresponder a um interesse do credor, digno de protecção legal.\nArtigo 399.º (Prestação de coisa futura)\nÉ admitida a prestação de coisa futura sempre que a lei não a proíba.\nArtigo 400.º (Determinação da prestação)\n1. A determinação da prestação pode ser confiada a uma ou outra das partes ou a terceiro; em qualquer dos casos deve ser feita segundo juízos de equidade, se outros critérios não tiverem sido estipulados.\n2. Se a determinação não puder ser feita ou não tiver sido feita no tempo devido, sê-lo-á pelo tribunal, sem prejuízo

Pode não ter subtitulo :

In [29]:
json_titulo = titulo_data[1]
livro = json_titulo["metadata"]['Livro']
titulo = json_titulo["metadata"]['Titulo']
titulo_text = json_titulo["text"]

# Regular expression to match "Artigo X.º (Title)"
matches = re.split(r"(SUBTÍTULO \d+)", titulo_text)[1:]  # Skip empty first element if any

if matches:

    # Structure the data
    subtitulo_data = [
        {
            "text": matches[i + 1].strip(),
            "metadata": {
                "Livro": livro,
                "Titulo": titulo,
                "Subtitulo": matches[i],
            },
        }
        for i in range(0, len(matches), 2)
    ]

else:
    # No matches found, return entire text with empty titulo metadata
    subtitulo_data = [
        {
            "text": titulo_text.strip(),
            "metadata": {
                "Livro": livro,
                "Titulo": titulo,
                "Subtitulo": "",
            },
        }
    ]

subtitulo_data

[{'text': 'Dos contratos em especial\nCAPÍTULO 1 Compra e venda\nSECÇÃO 1 Disposições gerais\nArtigo 874.º (Noção)\nCompra e venda é o contrato pelo qual se transmite a propriedade de uma coisa, ou outro direito, mediante um preço.\nARTIGO 875.º (Forma) Sem prejuízo do disposto em lei especial, o contrato de compra e venda de bens imóveis só é válido se for celebrado por escritura pública ou por documento particular autenticado.\nArtigo 876.º (Venda de coisa ou direito litigioso)\n1. Não podem ser compradores de coisa ou direito litigioso, quer directamente, quer por interposta pessoa, aqueles a quem a lei não permite que seja feita a cessão de créditos ou direitos litigiosos, conforme se dispõe no capítulo respectivo.\n2. A venda feita com quebra do disposto no número anterior, além de nula, sujeita o comprador, nos termos gerais, à obrigação de reparar os danos causados.\n3. A nulidade não pode ser invocada pelo comprador.\nArtigo 877.º (Venda a filhos ou netos)\n1. Os pais e avós nã

In [30]:
json_subtitulo = subtitulo_data[0]
livro = json_subtitulo["metadata"]['Livro']
titulo = json_subtitulo["metadata"]['Titulo']
subtitulo = json_subtitulo["metadata"]['Subtitulo']
subtitulo_text = json_subtitulo["text"]

# Split based on the chapter headings
matches = re.split(r"(CAPÍTULO \d+)", subtitulo_text)[1:]  # Skip empty first element if any

# Structure the data
capitulo_data = [
    {"text": matches[i + 1].strip(),
                 "metadata": {
                "Livro": livro,
                "Titulo": titulo,
                "Subtitulo": subtitulo,
                "capitulo": matches[i]
            },}
    for i in range(0, len(matches), 2)
]

capitulo_data

[{'text': 'Compra e venda\nSECÇÃO 1 Disposições gerais\nArtigo 874.º (Noção)\nCompra e venda é o contrato pelo qual se transmite a propriedade de uma coisa, ou outro direito, mediante um preço.\nARTIGO 875.º (Forma) Sem prejuízo do disposto em lei especial, o contrato de compra e venda de bens imóveis só é válido se for celebrado por escritura pública ou por documento particular autenticado.\nArtigo 876.º (Venda de coisa ou direito litigioso)\n1. Não podem ser compradores de coisa ou direito litigioso, quer directamente, quer por interposta pessoa, aqueles a quem a lei não permite que seja feita a cessão de créditos ou direitos litigiosos, conforme se dispõe no capítulo respectivo.\n2. A venda feita com quebra do disposto no número anterior, além de nula, sujeita o comprador, nos termos gerais, à obrigação de reparar os danos causados.\n3. A nulidade não pode ser invocada pelo comprador.\nArtigo 877.º (Venda a filhos ou netos)\n1. Os pais e avós não podem vender a filhos ou netos, se o

In [31]:
#for json_row in data:

json_artigo = capitulo_data[0]

livro = json_artigo["metadata"]['Livro']
titulo = json_artigo["metadata"]['Titulo']
subtitulo = json_artigo["metadata"]['Subtitulo']
capitulo = json_artigo["metadata"]['capitulo']
capitulo_text = json_artigo["text"]

# Regular expression to match "Artigo X.º (Title)"
pattern = r"(Artigo \d+\.º) \(([^)]+)\)"

# Split based on the article headings
matches = re.split(pattern, capitulo_text)[1:]  # Skip empty first element if any

# Structure the data
artigo_data = [
    {
        "text": matches[i + 2].strip(),
        "metadata": {
            "Livro": livro,
            "Titulo": titulo,
            "Subtitulo": subtitulo,
            "capitulo": capitulo,
            "artigo": matches[i],
            "titulo_artigo": matches[i + 1],
        },
    }
    for i in range(0, len(matches), 3)
]

artigo_data

[{'text': 'Compra e venda é o contrato pelo qual se transmite a propriedade de uma coisa, ou outro direito, mediante um preço.\nARTIGO 875.º (Forma) Sem prejuízo do disposto em lei especial, o contrato de compra e venda de bens imóveis só é válido se for celebrado por escritura pública ou por documento particular autenticado.',
  'metadata': {'Livro': 'LIVRO 2',
   'Titulo': 'TÍTULO 2',
   'Subtitulo': '',
   'capitulo': 'CAPÍTULO 1',
   'artigo': 'Artigo 874.º',
   'titulo_artigo': 'Noção'}},
 {'text': '1. Não podem ser compradores de coisa ou direito litigioso, quer directamente, quer por interposta pessoa, aqueles a quem a lei não permite que seja feita a cessão de créditos ou direitos litigiosos, conforme se dispõe no capítulo respectivo.\n2. A venda feita com quebra do disposto no número anterior, além de nula, sujeita o comprador, nos termos gerais, à obrigação de reparar os danos causados.\n3. A nulidade não pode ser invocada pelo comprador.',
  'metadata': {'Livro': 'LIVRO 2',


In [33]:
json_row = artigo_data[1]

livro = json_row["metadata"]['Livro']
titulo = json_row["metadata"]['Titulo']
subtitulo = json_row["metadata"]['Subtitulo']
capitulo = json_row["metadata"]['capitulo']
artigo = json_row["metadata"]['artigo']
artigo_title = json_row["metadata"]["titulo_artigo"]

row_text = json_row["text"]

# Regular expression to match numbers followed by a dot at the start or after a newline
pattern = r"(?:\n|^)(\d+)\. "

# Find all matches
matches = re.split(pattern, row_text)[1:]  # Skip empty first element if any

# Structure the data
grain_data = [
    {
        "text": matches[i + 1].strip(),
        "metadata": {            
                    "Livro": livro,
                    "Titulo": titulo,
                    "Subtitulo": subtitulo,
                    "capitulo": capitulo,
                    "artigo": artigo,
                    "titulo_artigo": artigo_title,
                    "numero": matches[i],
                    },
    }
    for i in range(0, len(matches), 2)
]

grain_data

[{'text': 'Não podem ser compradores de coisa ou direito litigioso, quer directamente, quer por interposta pessoa, aqueles a quem a lei não permite que seja feita a cessão de créditos ou direitos litigiosos, conforme se dispõe no capítulo respectivo.',
  'metadata': {'Livro': 'LIVRO 2',
   'Titulo': 'TÍTULO 2',
   'Subtitulo': '',
   'capitulo': 'CAPÍTULO 1',
   'artigo': 'Artigo 876.º',
   'titulo_artigo': 'Venda de coisa ou direito litigioso',
   'numero': '1'}},
 {'text': 'A venda feita com quebra do disposto no número anterior, além de nula, sujeita o comprador, nos termos gerais, à obrigação de reparar os danos causados.',
  'metadata': {'Livro': 'LIVRO 2',
   'Titulo': 'TÍTULO 2',
   'Subtitulo': '',
   'capitulo': 'CAPÍTULO 1',
   'artigo': 'Artigo 876.º',
   'titulo_artigo': 'Venda de coisa ou direito litigioso',
   'numero': '2'}},
 {'text': 'A nulidade não pode ser invocada pelo comprador.',
  'metadata': {'Livro': 'LIVRO 2',
   'Titulo': 'TÍTULO 2',
   'Subtitulo': '',
   'c